# Đánh giá 3 model Phi-3 trên Colab

Notebook này chạy benchmark Big Data cho 3 model Phi-3 local từ Hugging Face cache trên Google Drive. Luồng chạy ngắn gọn:

1. Mount Google Drive.
2. Clone repo `Hutaph/a-triple-of-lms` vào Colab.
3. Trỏ cache Hugging Face tới thư mục Drive đang chứa model.
4. Chạy `src/phi3.py` trên `data/bigdata_10_questions.json`.
5. Đọc summary và xem nhanh output mẫu.

Mặc định notebook dùng `LOAD_IN_4BIT = True` để hợp với Colab T4. Nếu GPU nhiều VRAM hơn, có thể tắt 4-bit.

## 1. Mount Google Drive

Drive được dùng để đọc model đã cache và lưu kết quả benchmark. Nếu thư mục cache khác `/content/drive/MyDrive/hf_cache`, chỉnh biến `HF_CACHE_ROOT` ở cell cấu hình.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Cấu hình

Ba model mặc định:

- `microsoft/Phi-3-mini-4k-instruct`
- `microsoft/Phi-3-small-8k-instruct`
- `microsoft/Phi-3-medium-4k-instruct`

Notebook chạy từ cache bằng `LOCAL_FILES_ONLY = True`, nên cần cache đã có sẵn trong Drive. Cấu trúc thường gặp là `hf_cache/hub/models--microsoft--...`.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
PROJECT_DIR = Path("/content/a-triple-of-lms")

DRIVE_ROOT = Path("/content/drive/MyDrive")
HF_CACHE_ROOT = DRIVE_ROOT / "hf_cache"
HF_HUB_CACHE = HF_CACHE_ROOT / "hub"
CACHE_DIR_FOR_SCRIPT = HF_HUB_CACHE if HF_HUB_CACHE.exists() else HF_CACHE_ROOT

DATA_FILE = PROJECT_DIR / "data" / "bigdata_10_questions.json"
OUTPUT_DIR = DRIVE_ROOT / "a-triple-of-lms_outputs" / "phi3"

MODEL_TO_RUN = "all"      # "mini", "small", "medium", hoặc "all"
BENCHMARK_LIMIT = 0       # 0 = chạy đủ 10 câu; đặt 2 hoặc 3 để smoke test nhanh
LOAD_IN_4BIT = True       # khuyến nghị cho Colab T4
LOCAL_FILES_ONLY = True   # dùng model trong Drive cache, không tải mới từ Hugging Face

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_ROOT / "transformers")
os.environ["PHI3_HF_CACHE_DIR"] = str(HF_HUB_CACHE)

print("Project dir:", PROJECT_DIR)
print("HF cache root:", HF_CACHE_ROOT)
print("HF hub cache:", HF_HUB_CACHE)
print("Cache dir for script:", CACHE_DIR_FOR_SCRIPT)
print("Output dir:", OUTPUT_DIR)

## 3. Clone hoặc cập nhật repo

Cell này lấy `data` và `src` mới nhất từ GitHub. Nếu repo đã tồn tại trong runtime Colab, notebook chỉ `git pull`.

In [ ]:
import subprocess

if PROJECT_DIR.exists():
    print("Repo đã tồn tại, đang cập nhật...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)
else:
    print("Đang clone repo...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

if not (PROJECT_DIR / "src" / "phi3.py").exists():
    raise FileNotFoundError("src/phi3.py not found. Push the new file to GitHub or use a repo version that contains it.")

## 4. Cài thư viện cần thiết

Colab thường đã có PyTorch. Cell này cài thêm Transformers, Accelerate và BitsAndBytes để load model Phi-3 từ cache, đặc biệt khi dùng 4-bit.

In [ ]:
!pip -q install -U "transformers>=4.43" accelerate bitsandbytes huggingface_hub tqdm python-dotenv

## 5. Kiểm tra GPU và cache model

Nếu dòng cache báo `MISSING`, hãy kiểm tra lại `HF_CACHE_ROOT` hoặc tải model vào Drive trước. Notebook đang cố tình dùng cache local để tránh tải lại nhiều GB mỗi lần reset runtime.

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_IDS = {
    "mini": "microsoft/Phi-3-mini-4k-instruct",
    "small": "microsoft/Phi-3-small-8k-instruct",
    "medium": "microsoft/Phi-3-medium-4k-instruct",
}

cache_base = HF_HUB_CACHE if HF_HUB_CACHE.exists() else HF_CACHE_ROOT
for key, model_id in MODEL_IDS.items():
    cache_name = "models--" + model_id.replace("/", "--")
    cache_path = cache_base / cache_name
    status = "OK" if cache_path.exists() else "MISSING"
    print(f"{key:6s} {status:7s} {cache_path}")

## 6. Chạy benchmark Phi-3

Script `src/phi3.py` đọc từng câu hỏi, sinh câu trả lời, chấm tự động nhẹ bằng `ground_truth`, và lưu JSON kết quả. Với T4, model medium có thể chậm; hãy đặt `BENCHMARK_LIMIT = 2` để thử nhanh trước khi chạy đủ.

In [ ]:
import subprocess

cmd = [
    "python", "src/phi3.py",
    "--model", MODEL_TO_RUN,
    "--data", str(DATA_FILE),
    "--output-dir", str(OUTPUT_DIR),
    "--cache-dir", str(CACHE_DIR_FOR_SCRIPT),
    "--device", "auto",
    "--torch-dtype", "auto",
]

if BENCHMARK_LIMIT:
    cmd += ["--limit", str(BENCHMARK_LIMIT)]
if LOAD_IN_4BIT:
    cmd += ["--load-in-4bit"]
if LOCAL_FILES_ONLY:
    cmd += ["--local-files-only"]

print("Command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 7. Xem summary

`avg_auto_score` là điểm tự động tham khảo dựa trên token/keyword so với `ground_truth`. Điểm này hữu ích để so sánh nhanh giữa model, nhưng vẫn nên đọc thủ công vài câu điểm thấp hoặc câu quan trọng.

In [ ]:
import json
import pandas as pd

summary_path = OUTPUT_DIR / "phi3_benchmark_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary_df = pd.DataFrame.from_dict(summary, orient="index")
summary_df

## 8. Xem nhanh output mẫu

Cell dưới đây đọc file tổng `phi3_benchmark_outputs.json` và hiển thị vài cột chính để kiểm tra câu trả lời, lỗi load/generate, latency và điểm tự động.

In [ ]:
outputs_path = OUTPUT_DIR / "phi3_benchmark_outputs.json"
rows = json.loads(outputs_path.read_text(encoding="utf-8"))
outputs_df = pd.DataFrame(rows)

cols = [
    "sample_id", "model_name", "difficulty", "category",
    "latency_s", "output_tokens", "auto_score", "error", "model_output",
]
outputs_df[cols].head(10)

## 9. Đường dẫn kết quả

Các file chính sau khi chạy:

- `phi3_benchmark_outputs.json`: toàn bộ output của cả 3 model.
- `Phi-3_Mini_4K_outputs.json`, `Phi-3_Small_8K_outputs.json`, `Phi-3_Medium_4K_outputs.json`: output theo từng model.
- `phi3_benchmark_summary.json`: summary để so sánh nhanh.

In [ ]:
print("Output directory:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("*.json")):
    print(path)